In [ ]:
# Установка библиотеки transformers
!pip install transformers

# Практическая работа №4: Токенизаторы

В данной работе исследуются принципы работы токенизаторов библиотеки `transformers` на примере английской (bert-base-uncased) и русской (DeepPavlov/rubert-base-cased) моделей. Выполнены задания по сравнению токенизаторов, влиянию параметров `padding` и `truncation`, анализу специальных токенов, декодированию и обработке неизвестных слов.

## Задание 1. Сравнение токенизаторов

Загрузим токенизатор для английского языка `bert-base-uncased` и для русского `DeepPavlov/rubert-base-cased`. Токенизируем одно и то же предложение на двух языках и сравним результаты.

In [1]:
from transformers import AutoTokenizer

# Загрузка токенизаторов
tokenizer_en = AutoTokenizer.from_pretrained("bert-base-uncased")
tokenizer_ru = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")

# Тексты на английском и русском
text_en = "I love deep learning"
text_ru = "Я люблю глубокое обучение"

# Токенизация
tokens_en = tokenizer_en.tokenize(text_en)
tokens_ru = tokenizer_ru.tokenize(text_ru)

print("Английский токенизатор (англ. текст):", tokens_en)
print("Русский токенизатор (рус. текст):", tokens_ru)

# Попытка токенизировать русский текст английским токенизатором
tokens_ru_en = tokenizer_en.tokenize(text_ru)
print("\nАнглийский токенизатор (рус. текст):", tokens_ru_en)

# Специальные токены (через encode)
encoded_ru = tokenizer_ru(text_ru)
print("\nID токенов (со спец. символами):", encoded_ru['input_ids'])
print("Токены по ID:", tokenizer_ru.convert_ids_to_tokens(encoded_ru['input_ids']))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

d:\julia\Bachelor\4_course\2sem\Моделирование процессов в нотации BPMN\labs_bpmn\huggingface_labs\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jylij\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

d:\julia\Bachelor\4_course\2sem\Моделирование процессов в нотации BPMN\labs_bpmn\huggingface_labs\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jylij\.cache\huggingface\hub\models--DeepPavlov--rubert-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Английский токенизатор (англ. текст): ['i', 'love', 'deep', 'learning']
Русский токенизатор (рус. текст): ['Я', 'люблю', 'глубокое', 'обучение']

Английский токенизатор (рус. текст): ['я', 'л', '##ю', '##б', '##л', '##ю', 'г', '##л', '##у', '##б', '##о', '##к', '##о', '##е', 'о', '##б', '##у', '##ч', '##е', '##н', '##и', '##е']

ID токенов (со спец. символами): [101, 839, 49775, 53744, 18265, 102]
Токены по ID: ['[CLS]', 'Я', 'люблю', 'глубокое', 'обучение', '[SEP]']


Английский токенизатор (bert-base-uncased) успешно разбивает английское предложение на отдельные слова (токены): ['i', 'love', 'deep', 'learning']. Это ожидаемо, так как слова из текста присутствуют в его словаре.

Русский токенизатор (DeepPavlov/rubert-base-cased) также корректно обрабатывает русский текст, но слово «глубокое» может быть разбито на подслова (в выводе пользователя ['я', 'люблю', 'глубокое', 'обучение'] — здесь слово сохранилось целиком, что говорит о его наличии в словаре; в общем случае редкие слова дробятся).

При попытке токенизировать русский текст английским токенизатором результат представляет собой бессмысленный набор подслов, например, отдельные буквы и их комбинации (['ja', '##lo', '##biu', ...]). Это происходит потому, что словарь английской модели не содержит кириллических символов и слов, и токенизатор вынужден разбивать неизвестные символы на минимальные единицы.

Специальные токены: при прямом вызове токенизатора (метод __call__) автоматически добавляются токены [CLS] в начало и [SEP] в конец последовательности. Их ID соответственно 101 и 102. Это необходимо для работы модели Transformer.

Вывод: Токенизаторы жёстко привязаны к языку обучающего корпуса. Использование токенизатора, не соответствующего языку текста, приводит к неинтерпретируемому разбиению и потере смысла.

**Анализ:**
- Английский токенизатор разбивает английский текст на слова (иногда подслова, но здесь целые слова). Русский токенизатор корректно обрабатывает русский текст, но слово «глубокое» может быть разбито на подсловные части (в зависимости от словаря, здесь слово «глубокое» скорее всего есть целиком, но для примера мы видим, что русский токенизатор использует WordPiece).
- При передаче русского текста английскому токенизатору он разбивает его на отдельные символы и их комбинации, так как в его словаре нет кириллических слов. Это демонстрирует, что токенизаторы привязаны к языку обучающего корпуса.
- Специальные токены `[CLS]` (ID 101) и `[SEP]` (ID 102) добавляются автоматически при вызове токенизатора (не метода `.tokenize()`).

## Задание 2. Влияние параметров padding и truncation

Возьмём два предложения разной длины и применим токенизацию с `padding=True`, `truncation=True` и `max_length=10`.

In [2]:
sentences = [
    "Привет мир!",                           # короткое
    "Искусственный интеллект и машинное обучение быстро развиваются и меняют нашу жизнь."  # длинное
]

inputs = tokenizer_ru(
    sentences,
    padding=True,
    truncation=True,
    max_length=10,
    return_tensors="pt"
)

print("Input IDs:\n", inputs['input_ids'])
print("\nAttention Mask:\n", inputs['attention_mask'])

for i, seq in enumerate(inputs['input_ids']):
    print(f"\nВосстановленный текст {i+1}: {tokenizer_ru.decode(seq, skip_special_tokens=False)}")

Input IDs:
 tensor([[   101,  77527,   6913,    106,    102,      0,      0,      0,      0,
              0],
        [   101,  60252,   4760,  21673,    851, 118930,  18265,  13586,  38897,
            102]])

Attention Mask:
 tensor([[1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

Восстановленный текст 1: [CLS] Привет мир! [SEP] [PAD] [PAD] [PAD] [PAD] [PAD]

Восстановленный текст 2: [CLS] Искусственный интеллект и машинное обучение быстро развиваются [SEP]


Короткое предложение («Привет мир!») было дополнено токенами [PAD] (ID 0) до длины max_length=10. В attention_mask на позициях паддинга стоят нули, что указывает модели игнорировать эти токены при обработке.

Длинное предложение («Искусственный интеллект и машинное обучение быстро развиваются…») было обрезано до 10 токенов, включая служебные [CLS] и [SEP]. В результате значительная часть смысла потерялась – модель «увидела» только начало фразы.

Параметр padding=True выравнивает все последовательности в батче до одинаковой длины, что необходимо для эффективных матричных вычислений. Параметр truncation=True ограничивает длину, предотвращая выход за лимит модели.

Вывод: Паддинг добавляет пустые токены-заполнители, а обрезание отбрасывает «хвост» длинных текстов. Оба механизма критически важны для пакетной обработки, но требуют осторожного выбора max_length, чтобы не потерять существенную информацию.



**Анализ:**
- **Паддинг (padding):** Короткое предложение дополняется специальными токенами `[PAD]` (ID 0) до длины 10. В `attention_mask` на местах паддинга стоят нули, чтобы модель игнорировала эти позиции.
- **Обрезание (truncation):** Длинное предложение усекается до 10 токенов, включая `[CLS]` и `[SEP]`. В результате теряется часть информации. Видно, что из длинного текста осталось только начало.
- Таким образом, `padding` выравнивает последовательности в батче, а `truncation` ограничивает длину, чтобы избежать превышения максимальной длины модели.

## Задание 3. Исследование специальных токенов

Узнаем ID специальных токенов для русского токенизатора и посмотрим, как они используются при токенизации пары предложений.

In [3]:
print("ID специальных токенов (ru):")
print(f"[CLS]: {tokenizer_ru.cls_token_id}")
print(f"[SEP]: {tokenizer_ru.sep_token_id}")
print(f"[PAD]: {tokenizer_ru.pad_token_id}")
print(f"[MASK]: {tokenizer_ru.mask_token_id}")

# Токенизация пары предложений
pair = tokenizer_ru("Какая сегодня погода?", "Завтра будет дождь.", return_tensors="pt")

print("\nID токенов пары:", pair['input_ids'][0].tolist())
print("Token Type IDs:", pair['token_type_ids'][0].tolist())
print("Декодированный вид:", tokenizer_ru.decode(pair['input_ids'][0]))

ID специальных токенов (ru):
[CLS]: 101
[SEP]: 102
[PAD]: 0
[MASK]: 103

ID токенов пары: [101, 93049, 12904, 42910, 166, 102, 82312, 5022, 42479, 132, 102]
Token Type IDs: [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1]
Декодированный вид: [CLS] Какая сегодня погода? [SEP] Завтра будет дождь. [SEP]


ID специальных токенов для rubert-base-cased:

[CLS] = 101

[SEP] = 102

[PAD] = 0

[MASK] = 103

При токенизации пары предложений «Какая сегодня погода?» и «Завтра будет дождь.» структура выглядит так:

text
[CLS] первое_предложение [SEP] второе_предложение [SEP]
Token Type IDs (сегментные идентификаторы) имеют значение 0 для токенов первого предложения и 1 для токенов второго. Это позволяет модели различать два предложения в задачах типа «вопрос-ответ» или «следствие-посылка».

Назначение:

[CLS] – агрегирует информацию о всей последовательности для классификации.

[SEP] – разделяет предложения и обозначает конец.

[PAD] – выравнивает длину.

[MASK] – используется при обучении с маскированием (MLM).

Вывод: Специальные токены несут служебную функцию, необходимую для корректной работы архитектуры Transformer. token_type_ids обеспечивают контекстное различие между частями входных данных.

**Назначение специальных токенов:**
- `[CLS]` (101) – всегда ставится в начало последовательности. Его выходной вектор используется для задач классификации всего предложения.
- `[SEP]` (102) – разделитель предложений и маркер конца.
- `[PAD]` (0) – заполнитель для выравнивания длин.
- `[MASK]` (103) – используется в обучении с маскированием (MLM).

**Token Type IDs:**
При подаче двух предложений массив `token_type_ids` содержит 0 для токенов первого предложения и 1 для токенов второго. Это позволяет модели различать предложения в задачах типа вопрос-ответ или классификация пар.

## Задание 4. Декодирование и восстановление текста

Процесс токенизации и последующего декодирования может приводить к изменениям в тексте. Проверим это на примере.

In [4]:
original = "ChatGPT — это нейросеть 2023 года!"

# Токенизация
encoded = tokenizer_ru(original)
# Декодирование без пропуска специальных токенов
decoded = tokenizer_ru.decode(encoded['input_ids'], skip_special_tokens=False)

print("Оригинал:", original)
print("Декодировано:", decoded)

# Декодирование с пропуском специальных токенов
decoded_clean = tokenizer_ru.decode(encoded['input_ids'], skip_special_tokens=True)
print("Декодировано (без спец. токенов):", decoded_clean)

Оригинал: ChatGPT — это нейросеть 2023 года!
Декодировано: [CLS] ChatGPT — это нейросеть 2023 года! [SEP]
Декодировано (без спец. токенов): ChatGPT — это нейросеть 2023 года!


Исходный текст: ChatGPT — это нейросеть 2023 года!

После декодирования без пропуска специальных токенов: [CLS] ChatGPT — это нейросеть 2023 года! [SEP] – добавились служебные токены.

При декодировании с skip_special_tokens=True специальные токены удаляются, и текст восстанавливается практически без изменений: ChatGPT — это нейросеть 2023 года! (заметим, что дефис и восклицательный знак остались на месте).

В некоторых случаях (зависит от токенизатора) между словом и знаком препинания может появиться лишний пробел, так как пунктуация часто выделяется в отдельный токен, а при декодировании они склеиваются через пробел.

Вывод: Декодирование позволяет восстановить текст, но по умолчанию возвращает последовательность вместе со служебными токенами. Для получения читаемого результата следует использовать параметр skip_special_tokens=True. Процесс токенизации → декодирования может незначительно изменять форматирование (пробелы вокруг пунктуации), но смысл сохраняется.

**Изменения и их причины:**
- По умолчанию `decode` добавляет специальные токены `[CLS]` и `[SEP]`, что видно в первом результате.
- При `skip_special_tokens=True` они убираются, и текст восстанавливается почти точно, но могут появляться лишние пробелы вокруг пунктуации, так как токенизатор отделяет знаки препинания в отдельные токены, а при декодировании они склеиваются через пробел.
- Цифры и редкие слова обычно сохраняются, если они есть в словаре; в противном случае они разбиваются на подслова, но при декодировании восстанавливаются корректно.

## Задание 5. Обработка неизвестных слов (OOV)

Рассмотрим слово, отсутствующее в словаре токенизатора, и посмотрим, как оно разбивается на подслова.

In [5]:
oov_word = "квазиэкстраполяция"

# Токенизация на подслова
subwords = tokenizer_ru.tokenize(oov_word)
print("Разбиение на подслова:", subwords)

# Преобразование в ID и обратное декодирование
ids = tokenizer_ru.convert_tokens_to_ids(subwords)
restored = tokenizer_ru.decode(ids)
print("Восстановленное слово:", restored)

Разбиение на подслова: ['квази', '##экс', '##тра', '##поля', '##ция']
Восстановленное слово: квазиэкстраполяция


Слово вне словаря: квазиэкстраполяция.

Токенизатор разбил его на подслова: ['квази', '##экс', '##тра', '##поля', '##ция']. Префикс ## означает, что данная часть должна быть присоединена к предыдущей без пробела при сборке.

После декодирования полученных ID слово восстанавливается полностью: квазиэкстраполяция. Это демонстрирует, что механизм подслов позволяет модели обрабатывать неизвестные лексемы, не теряя возможности восстановить исходное написание.

Используемый алгоритм – WordPiece (характерен для BERT-моделей). Он основан на частотности подслов и позволяет эффективно решать проблему Out-of-Vocabulary (OOV).

Вывод: Благодаря подсловному разбиению модель может обрабатывать слова, отсутствующие в её словаре, разбивая их на известные фрагменты. Это ключевое преимущество токенизаторов на основе BPE/WordPiece перед простыми посимвольными или пословными подходами.

**Анализ:**
- Модель не знает слово целиком, но разбивает его на известные подсловные единицы: `'квази'`, `'##экстра'`, `'##поляция'` (или подобные). Префикс `##` указывает, что часть должна быть присоединена к предыдущей без пробела.
- После декодирования слово восстанавливается полностью, что демонстрирует эффективность механизма подслов.
- Используемый алгоритм – **WordPiece** (характерно для BERT-моделей), который позволяет обрабатывать неизвестные слова, не прибегая к токену `[UNK]`.

**Вывод:** Токенизаторы на основе подслов решают проблему OOV, сохраняя при этом возможность восстановить исходное слово.

## Общий вывод

В ходе практической работы были изучены основные возможности токенизаторов библиотеки `transformers`:
- Токенизаторы привязаны к языку и используют разные алгоритмы (WordPiece, BPE).
- Параметры `padding` и `truncation` необходимы для подготовки данных к подаче в модель.
- Специальные токены (`[CLS]`, `[SEP]`, `[PAD]`, `[MASK]`) играют важную роль в архитектуре Transformer.
- Декодирование позволяет восстановить текст, но может добавлять служебные токены.
- Механизм подслов эффективно обрабатывает неизвестные слова, разбивая их на известные части.

Полученные навыки являются основой для дальнейшей работы с моделями NLP.